In [1]:
# 14.2.1 Cài đặt và Import thư viện
from collections import Counter, defaultdict
from typing import List, Dict, Literal, Union

import re
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings("ignore")

CACHE_DIR = "./cache"

In [2]:
# 14.2.2 Đọc và khám phá bộ dữ liệu
ds = load_dataset("UniverseTBD/arxiv-abstracts-large") # Đã sửa lỗi: thay ' ' bằng '-' theo quy tắc đặt tên của Hugging Face
ds

README.md:   0%|          | 0.00/810 [00:00<?, ?B/s]

arxiv-metadata-oai-snapshot.json:   0%|          | 0.00/3.82G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2292057 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})

In [ ]:
# in ra màn hình 3 bản ghi đầu tiên của bộ dữ liệu
for i in range(3):
  print(f"Example {i+1}:")
  print(ds['train'][i]['abstract'])
  print(ds['train'][i]['categories'])
  print("---" * 20)

Example 1:
  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. The region of phase space is
specified in which the calculation is most reliable. Good agreement is
demonstrated with data from the Fermilab Tevatron, and predictions are made for
more detailed tests with CDF and DO data. Predictions are shown for
distributions of diphoton pairs produced at the energy of the Large Hadron
Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs
boson are contrasted with those produced from QCD processes at the LHC, showing
that enhanced sensitivity to the signal can be obtained with judicious
selection of events.

hep-p

In [ ]:
# in toàn bộ các categories của abstract để xem xét các nhãn này có phù hợp với bài toán phân loại hay không
all_categories = ds['train']['categories']
print(set(all_categories))

{'quant-ph cond-mat.mes-hall cond-mat.other physics.optics', 'cs.OH math.DS', 'physics.flu-dyn math-ph math.DS math.MP nlin.CD', 'cs.DC cs.PF cs.SY', 'cs.NE cs.ET nlin.AO q-bio.NC', 'astro-ph nlin.CD physics.geo-ph', 'cond-mat.dis-nn cs.SI math-ph math.MP physics.soc-ph', 'astro-ph.EP physics.ao-ph physics.flu-dyn physics.space-ph', 'cs.CV cs.CR cs.LG cs.NE stat.ML', 'cs.CL cs.AI cs.HC cs.IT math.IT', 'physics.med-ph eess.SP math.SP q-bio.TO', 'nlin.CD cond-mat.stat-mech physics.class-ph', 'cs.LG cs.AI math.OC math.ST stat.ML stat.TH', 'cond-mat.str-el cond-mat.mes-hall cond-mat.mtrl-sci hep-th quant-ph', 'eess.SP nlin.AO', 'physics.atom-ph physics.app-ph physics.data-an', 'physics.space-ph astro-ph.EP astro-ph.IM physics.comp-ph physics.plasm-ph', 'hep-ph astro-ph.IM physics.ins-det', 'cs.DC cs.DS stat.ML', 'quant-ph hep-ph hep-th nucl-th', 'physics.ao-ph cs.LG stat.AP stat.CO stat.ME', 'cs.MA cs.IT cs.LG math.IT', 'cs.LO cs.CR cs.DS', 'physics.atm-clus nucl-th physics.optics quant-ph

In [ ]:
# để quan sát số lượng primary categories trong bộ dữ liệu, chúng ta sẽ đếm số lượng các nhãn này và in ra như sau
all_categories = ds['train']['categories']
category_set = set()

# Collect unique labels
for category in all_categories:
    parts = category.split(' ')
    for part in parts:
        topic = part.split('.')[0]
        category_set.add(topic)

# Sort the labels and print them
sorted_categories = sorted(list(category_set), key=lambda x: x.lower())

print(f'There are {len(sorted_categories)} unique primary categories in the dataset:')

for category in sorted_categories:
    print(category)


There are 38 unique primary categories in the dataset:
acc-phys
adap-org
alg-geom
ao-sci
astro-ph
atom-ph
bayes-an
chao-dyn
chem-ph
cmp-lg
comp-gas
cond-mat
cs
dg-ga
econ
eess
funct-an
gr-qc
hep-ex
hep-lat
hep-ph
hep-th
math
math-ph
mtrl-th
nlin
nucl-ex
nucl-th
patt-sol
physics
plasm-ph
q-alg
q-bio
q-fin
quant-ph
solv-int
stat
supr-con


In [ ]:
# load 1000 samples with single label belonging to specific categories
# Lấy 1000 samples, các samples này có primary là một trong 5 categories sau: astro-ph, cond-mat, cs, math, physics
samples = []
CATEGORIES_TO_SELECT = ['astro-ph', 'cond-mat', 'cs', 'math', 'physics']

for s in ds['train']:
    if len(s['categories'].split(' ')) != 1:
        continue

    cur_category = s['categories'].strip().split('.')[0]
    if cur_category not in CATEGORIES_TO_SELECT:
        continue

    samples.append(s)

    if len(samples) >= 1000:
        break

print(f"Number of samples: {len(samples)}")  # Number of samples: 1000


Number of samples: 1000


In [ ]:
# 14.2.3 Tiền xử lý dữ liệu
# Tiền xử lý dữ liệu đặc trưng
preprocessed_samples = []
for s in samples:
    abstract = s['abstract']

    # Remove \n characters in the middle and leading/trailing spaces
    abstract = abstract.strip().replace("\n", " ")

    # Remove special characters
    abstract = re.sub(r'[^\w\s]', '', abstract)

    # Remove digits
    abstract = re.sub(r'\d+', '', abstract)

    # Remove extra spaces
    abstract = re.sub(r'\s+', ' ', abstract).strip()

    # Convert to lower case, Đưa toàn bộ chữ cái về dạng thường để thống nhất.
    abstract = abstract.lower()

    # for the label, we only keep the first part, chuẩn hóa nhãn
    parts = s['categories'].split(' ')
    category = parts[0].split('.')[0]

    preprocessed_samples.append({
        "text": abstract,
        "label": category
    })


In [ ]:
# cần tạo hai dictionaries để lưu trữ các primary categories dưới dạng số và ngược lại
label_to_id = {label: i for i, label in enumerate(sorted_categories)}
id_to_label = {i: label for i, label in enumerate(sorted_categories)}
# Print label to ID mapping
print("Label to ID mapping:")
for label, id_ in label_to_id.items():
    print(f"{label} --> {id_}")

Label to ID mapping:
acc-phys --> 0
adap-org --> 1
alg-geom --> 2
ao-sci --> 3
astro-ph --> 4
atom-ph --> 5
bayes-an --> 6
chao-dyn --> 7
chem-ph --> 8
cmp-lg --> 9
comp-gas --> 10
cond-mat --> 11
cs --> 12
dg-ga --> 13
econ --> 14
eess --> 15
funct-an --> 16
gr-qc --> 17
hep-ex --> 18
hep-lat --> 19
hep-ph --> 20
hep-th --> 21
math --> 22
math-ph --> 23
mtrl-th --> 24
nlin --> 25
nucl-ex --> 26
nucl-th --> 27
patt-sol --> 28
physics --> 29
plasm-ph --> 30
q-alg --> 31
q-bio --> 32
q-fin --> 33
quant-ph --> 34
solv-int --> 35
stat --> 36
supr-con --> 37


In [ ]:
# Cuối cùng, chúng ta sẽ chia dữ liệu trên thành 2 tập: tập huấn luyện
# (train) và tập kiểm tra (test) với tỉ lệ 80% cho tập huấn luyện và 20% cho
# tập kiểm tra. Chúng ta sẽ sử dụng hàm train_test_split từ thư viện
# sklearn để thực hiện việc này
X_full = [sample['text'] for sample in preprocessed_samples]
y_full = [label_to_id[sample['label']] for sample in preprocessed_samples]

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")



Training samples: 800
Test samples: 200


In [ ]:
# 14.2.4 Mã hóa văn bản
# Bag-of-Words (BOW)
docs = [
    "I am going to school to study for the final exam.",
    "The weather is nice today and I feel happy.",
    "I love programming in Python and exploring new libraries.",
    "Data science is an exciting field with many opportunities.",
]

bow = CountVectorizer()
vectors = bow.fit_transform(docs)

for i, vec in enumerate(vectors):
    print(f"Document {i+1}: {vec.toarray()}")


Document 1: [[1 0 0 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 2 0 0 0]]
Document 2: [[0 0 1 0 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0]]
Document 3: [[0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 1 0 0 1 1 0 0 0 0 0 0 0 0]]
Document 4: [[0 1 0 1 0 1 0 0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 1 0 0 0 0 0 1]]


In [ ]:
# TF-IDF
docs = [
    "I am going to school to study for the final exam.",
    "The weather is nice today and I feel happy.",
    "I love programming in Python and exploring new libraries.",
    "Data science is an exciting field with many opportunities.",
]

vectorizer = TfidfVectorizer()
tfidf_vectors = vectorizer.fit_transform(docs)

for i, vec in enumerate(tfidf_vectors):
    print(f"TF-IDF for Document {i+1}:")
    print(vec.toarray())


TF-IDF for Document 1:
[[0.29333722 0.         0.         0.         0.29333722 0.
  0.         0.         0.         0.29333722 0.29333722 0.29333722
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.29333722
  0.         0.29333722 0.23127044 0.58667444 0.         0.
  0.        ]]
TF-IDF for Document 2:
[[0.         0.         0.30091213 0.         0.         0.
  0.         0.38166888 0.         0.         0.         0.
  0.38166888 0.         0.30091213 0.         0.         0.
  0.         0.38166888 0.         0.         0.         0.
  0.         0.         0.30091213 0.         0.38166888 0.38166888
  0.        ]]
TF-IDF for Document 3:
[[0.         0.         0.2855815  0.         0.         0.
  0.36222393 0.         0.         0.         0.         0.
  0.         0.36222393 0.         0.36222393 0.36222393 0.
  0.36222393 0.         0.         0.36222393 0.36222393 0.
  0.         0.         0.         0.

In [ ]:
# Sentence Embeddings
class EmbeddingVectorizer:
    def __init__(
        self,
        model_name: str = "intfloat/multilingual-e5-base",
        normalize: bool = True,
    ):
        self.model = SentenceTransformer(model_name)
        self.normalize = normalize

    def _format_inputs(
        self,
        texts: List[str],
        mode: Literal["query", "passage"],
    ) -> List[str]:
        if mode not in {"query", "passage"}:
            raise ValueError("Mode must be either 'query' or 'passage'")
        return [f"{mode}: {text.strip()}" for text in texts]

    def transform(
        self,
        texts: List[str],
        mode: Literal["query", "passage"] = "query",
    ) -> List[List[float]]:
        inputs = self._format_inputs(texts, mode)
        embeddings = self.model.encode(
            inputs,
            normalize_embeddings=self.normalize,
        )
        return embeddings.tolist()

    def transform_numpy(
        self,
        texts: List[str],
        mode: Literal["query", "passage"] = "query",
    ) -> np.ndarray:
        return np.array(self.transform(texts, mode=mode))

